In [29]:
%cd ..

/home/gmcirco/Documents


In [53]:
from datetime import date
from enum import Enum
from src.base import pdf_to_png, extract_from_doc
from src.utils import pretty

from pydantic import BaseModel, Field


class PageCategory(str, Enum):
  PRINTED_EMAIL = "Printed Email"
  SCANNED_INTAKE = "Scanned Intake Sheet"
  LOAN_DOCUMENT = "Loan Document"
  INVOICE = "Invoice"
  OTHER = "Other Document Type"


class PageType(BaseModel):
  page_type: PageCategory = Field(description="One of the allowed page types")
  reasoning: str = Field(description="Reasoning for the page type classification in 10 words or less")
  pages: list[int] = Field(description="Page number(s) assigned to this page type")

class DocumentSegmentor(BaseModel):
    segments: list[PageType] = Field(description="List of page type segments in the document")
    

In [36]:
img = pdf_to_png("/home/gmcirco/Documents/Book/practical-document-analysis-with-llms/documents/invoices/invoice_2.pdf")
document_segmentor = extract_from_doc(img, DocumentSegmentor)
pretty(document_segmentor)


{'segments': [{'page_type': <PageCategory.OTHER: 'Other Document Type'>,
               'reasoning': 'Regulatory filing, tariff schedules, cost '
                            'analyses, and supporting documents.',
               'pages': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]},
              {'page_type': <PageCategory.INVOICE: 'Invoice'>,
               'reasoning': 'Supplier invoices documenting materials and '
                            'purchase amounts.',
               'pages': [13,
                         14,
                         15,
                         16,
                         17,
                         18,
                         19,
                         20,
                         21,
                         22,
                         23,
                         24,
                         25,
                         26]}]}


In [41]:
document_segmentor.segments[1].page_type.value


'Invoice'

In [42]:

from pymupdf import pymupdf


def segment_document(pdf_path: str, segmentation_result: dict) -> dict:
  """Slices a PDF into image-byte lists grouped by PageCategory."""
  with pymupdf.open(pdf_path) as pdf:
    return {
        seg["page_type"]: [
            pdf[p - 1].get_pixmap(dpi=150).tobytes("png") for p in seg["pages"]
        ]
        for seg in segmentation_result["segments"]
    }

In [ ]:
# this splits the full doc into only single doc types
segmented_document = segment_document("documents/invoices/invoice_2.pdf", document_segmentor.model_dump())

dict_keys([<PageCategory.OTHER: 'Other Document Type'>, <PageCategory.INVOICE: 'Invoice'>])


In [64]:
class Invoice(BaseModel):
    invoice_number: str
    invoice_date: date
    invoice_due_date: date
    total_amount: float


# this defines a single item description
class ItemDescription(BaseModel):
    product_number: str
    description: str

# this extracts a list of item descriptions, as defined aboive
class InvoiceWithAnnotations(BaseModel):
    invoice_number: str = Field(description="The invoice number")
    invoice_date: date = Field(description="The date the invoice was issued in format YYYY-MM-DD")
    invoice_due_date: date = Field(description="The date the invoice is due in format YYYY-MM-DD")
    total_amount: float = Field(description="The total amount of the invoice")
    item_description: list[ItemDescription] = Field(description="Product number and description of each item")

class InvoiceList(BaseModel):
    invoices: list[InvoiceWithAnnotations] = Field(description="A list of invoices")

In [65]:
invoice = extract_from_doc(segmented_document["Invoice"], InvoiceList, role_prompt="Return all dates in the format YYYY-MM-DD")

In [66]:
pretty(invoice)

{'invoices': [{'invoice_number': '4189345',
               'invoice_date': datetime.date(2018, 3, 13),
               'invoice_due_date': datetime.date(2018, 4, 13),
               'total_amount': 299.74,
               'item_description': [{'product_number': 'FOM634CCS70663',
                                     'description': '6X3/4 CC S70 SDL F/IPS '
                                                    'PVC S70-603 FORD'},
                                    {'product_number': 'FOM434CCS70450',
                                     'description': '4X3/4 CC S70 SDL F/IPS '
                                                    'PVC S70-403 FORD'},
                                    {'product_number': 'FOM34F1000GNL',
                                     'description': '3/4" CORP STOP CCXGJC NO '
                                                    'LEAD F1000-3-G-NL'}]},
              {'invoice_number': '1243615',
               'invoice_date': datetime.date(2018, 2, 22),
               '